In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,root_mean_squared_log_error
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split 


# Data loading

In [2]:

results_calculation = {}
ignore_filters = [
  "Source",
  "Destination",
  'ts_gps_source',
  'ts_gps_destination']
# Transform categorical features
categorical_features = [
  "SNR",
]


In [3]:
def preprocess_nan_values(dataFrame,filters = []):
    """Replace NaN in categorical columns with empty string and maintain proper dtypes"""
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            if pd.api.types.is_string_dtype(dataFrame[col]) or pd.api.types.is_object_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].fillna('Unknown').astype(str)
            elif pd.api.types.is_integer_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].abs()
            else:
                dataFrame[col] = dataFrame[col]
        else:
            if pd.api.types.is_datetime64_any_dtype(dataFrame[col]):
                # Convert datetime to float (timestamp in seconds)
                dataFrame[col] = dataFrame[col].astype('int64') / 1e9
            else:
                dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame



def preprocess_category_values(dataFrame,filters = []):
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            dataFrame[col] = dataFrame[col].fillna('Unknown').astype('category')
        else:
            if pd.api.types.is_datetime64_any_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].astype('int64') / 1e9
            else:
                dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame

In [4]:
cellular_df = preprocess_nan_values(pd.read_csv('cellular_df.csv'),ignore_filters)
cellular_df.drop(columns=ignore_filters,inplace=True)


## Feature Segregation and Importances

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
def plot_feature_distribution(dataFrame, features,target):
    plt.figure(figsize=(15, 10))
    # Plot relationships between key network metrics
    # Accept both float and category dtypes for plotting
    print("Plotting feature distribution for features:", features,dataFrame['SNR'])
    valid_features = []
    for f in features:
        if isinstance(dataFrame[f], pd.Float64Dtype) or isinstance(dataFrame[f], pd.Categorical):
            valid_features.append(f)
        else:
            valid_features = features
    sns.pairplot(dataFrame[valid_features])
    plt.suptitle('Feature Relationships', y=1.02)
    plt.show()

    # 2. Feature Importance Analysis
    # Preprocessing
    df = dataFrame.copy()

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    df_imputed = pd.DataFrame(imputer.fit_transform(df.select_dtypes(include=['float64'])), 
                            columns=df.select_dtypes(include=['float64']).columns)

    X = df_imputed.drop(columns=[target, 'id'])  # Exclude identifiers
    y = df_imputed[target]

    # Train Random Forest model
    model = RandomForestRegressor(n_estimators=150, random_state=42)
    model.fit(X, y)

    # Get feature importances
    feature_importances = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    # Plot feature importance
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances)
    plt.title('Feature Importance for '+target.title()+' Prediction')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.show()

    # Display top features
    print("Top 10 Important Features:")
    print(feature_importances.head(10))
    
show_feature_distribution = False
if show_feature_distribution:
    plot_feature_distribution(cellular_df.head(10000), categorical_features,'pdr')

# Data preparation

In [6]:
# Set up category values to categorical features
cellular_df = preprocess_category_values(cellular_df,categorical_features)

# Split inputs and targets
X = cellular_df.drop(columns=['throughput'])
y = cellular_df['throughput']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)



# Prediction algorithm using Decision Tree Regressor

In [7]:
dt_regressor = DecisionTreeRegressor(
    max_depth=5,          # Control tree depth to prevent overfitting
    min_samples_split=10, # Minimum samples required to split a node
    random_state=42       # For reproducibility
)

dt_regressor.fit(X_train, y_train)  # Train on the training data
y_test_dt = dt_regressor.predict(X_test)
y_test_shifted = y_test - y_test.min() + 1
y_test_dt_shifted = y_test_dt - y_test.min() + 1

# Compute error metric for Decision Tree Regressor

In [8]:

r2 = r2_score(y_test_shifted, y_test_dt_shifted)
rmse = root_mean_squared_log_error(y_test_shifted, y_test_dt_shifted)
mae = mean_absolute_error(y_test_shifted, y_test_dt_shifted)
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

results_calculation['Decision Tree Regressor'] = {'r2':r2, 'rmse': rmse, 'mae':mae}


R²: 0.9994
RMSE: 0.0086
MAE: 2.6127


# Prediction using XGBoost

In [11]:

model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        enable_categorical=True
    )

model.fit(X_train, y_train)

y_pred_xg = model.predict(X_test)
reformed_pred=[]
for val in y_pred_xg:
    if val < 0:
        reformed_pred.append(0)
    else:
        reformed_pred.append(val)
        
y_pred_xg = reformed_pred
y_test = y_test - y_test.min() + 1
# y_pred_xg_shifted = y_pred_xg - y_test.min() + 1


In [10]:

r2_xg = r2_score(y_test, y_pred_xg)
rmse_xg =root_mean_squared_log_error(y_test, y_pred_xg)
mae_xg = mean_absolute_error(y_test, y_pred_xg)
print(f"R²: {r2_xg:.4f}")
print(f"RMSE: {rmse_xg:.3f}")
print(f"MAE: {mae_xg:.4f}")

results_calculation['XGBoost'] = {'r2':r2_xg, 'rmse': rmse_xg, 'mae':mae_xg}


ValueError: Root Mean Squared Logarithmic Error cannot be used when targets contain values less than or equal to -1.

# Cat Boost Algorithm Prediction

In [ ]:
print(X_train['SNR'].head(3))

In [ ]:


cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features]

from catboost import Pool

#POOLING
train_pool = Pool(
    data=preprocess_nan_values(X_train,categorical_features),
    label=y_train,
    cat_features=categorical_features
)

test_pool = Pool(
    data=preprocess_nan_values(X_test,categorical_features),
    label=y_test,
    cat_features=categorical_features
)

# Model for CatBoostAlgorithm 
model_cat = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=4,
    verbose=1000,
    cat_features=cat_features_indices,
    random_seed=42,
    early_stopping_rounds=10,
    l2_leaf_reg=10,
    loss_function='RMSE',
    allow_writing_files=False
)



model_cat.fit(train_pool,eval_set=test_pool)

pred_cat = model_cat.predict(test_pool)

reformed_pred=[]
for val in pred_cat:
    if val < 0:
        reformed_pred.append(0)
    else:
        reformed_pred.append(val)
        
pred_cat = reformed_pred





In [ ]:
r2_cat = r2_score(y_test, pred_cat)
rmse_cat = root_mean_squared_log_error(y_test, pred_cat)
mae_cat = mean_absolute_error(y_test, pred_cat)

print(f"R²: {r2_cat:.4f}")
print(f"RMSE: {rmse_cat:.4f}")
print(f"MAE: {mae_cat:.4f}")

results_calculation['CatBoost'] = {'r2':r2_cat, 'rmse':  rmse_cat, 'mae':mae_cat}


## CREATE A PLOTTING GRAPH .

In [ ]:
import plotly.graph_objects as go

model_names = list(results_calculation.keys())
r2_values = [results_calculation[model]['r2'] for model in model_names]
rmse_values = [results_calculation[model]['rmse'] for model in model_names]
mae_values = [results_calculation[model]['mae'] for model in model_names]

# Creating the bar chart
fig = go.Figure()

# Adding R-Square (R²) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=r2_values,
    name='R² Error',
    marker_color='blue'
))

# Adding Root Mean Squared Error (RMSE) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=rmse_values,
    name='RMSE',
    marker_color='green'
))

# Adding MAE bars
fig.add_trace(go.Bar(
    x=model_names,
    y=mae_values,
    name='MAE',
    marker_color='red'
))

# Updating layout
fig.update_layout(
    title='Comparison of Model Performance Metrics',
    xaxis_title='Models',
    yaxis_title='Metric Values',
    barmode='group',
    legend_title='Metrics',
    template='plotly_white'
)

# Show the plot
fig.show()